# __Carnot Data Engineer Assignment – Satellite Intelligence__
### __Author: Neha Pattankar__
### Pipeline: Ingest → Clean → Join → Analyse → Export

In [148]:
#Importing Libraries
import numpy as np 
import pandas as pd 

In [149]:
##Loading the data
metadata = pd.read_csv('/Users/nehapattankar/Documents/Carnot_Tech/data/parcel_metadata.csv')
readings = pd.read_csv('/Users/nehapattankar/Documents/Carnot_Tech/data/parcel_readings.csv')

## Basic Inspection

In [150]:
#Metadata info
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   parcel_id      28 non-null     str    
 1   mill_id        28 non-null     str    
 2   crop_type      28 non-null     str    
 3   sowing_date    28 non-null     str    
 4   area_hectares  28 non-null     float64
dtypes: float64(1), str(4)
memory usage: 1.2 KB


In [151]:
#Readings info
readings.info()

<class 'pandas.DataFrame'>
RangeIndex: 3447 entries, 0 to 3446
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   parcel_id      3447 non-null   str    
 1   date           3447 non-null   str    
 2   ndvi_value     3447 non-null   float64
 3   temperature_c  3447 non-null   float64
 4   rainfall_mm    3447 non-null   float64
 5   sensor_status  3310 non-null   str    
dtypes: float64(3), str(3)
memory usage: 161.7 KB


In [152]:
#Checking for null values in metadata
metadata.isnull().sum()

parcel_id        0
mill_id          0
crop_type        0
sowing_date      0
area_hectares    0
dtype: int64

In [153]:
#checking for null values in readings
readings.isnull().sum()

parcel_id          0
date               0
ndvi_value         0
temperature_c      0
rainfall_mm        0
sensor_status    137
dtype: int64

### Data Quality

In [154]:
#Checking the metadata
metadata.head()

,parcel_id,mill_id,crop_type,sowing_date,area_hectares
0,PARCEL_001,MILL_NORTH,sugarcane,2026-02-10,9.03
1,PARCEL_002,MILL_SOUTH,sugarcane,2026-01-16,8.97
2,PARCEL_003,MILL_NORTH,soybean,2026-02-13,5.35
3,PARCEL_004,MILL_NORTH,sugarcane,2026-01-02,3.18
4,PARCEL_005,MILL_NORTH,soybean,2026-02-08,2.79


In [155]:
#Checking the Readings
readings.head()

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
0,PARCEL_018,16/05/2026,0.595,15.4,0.0,error
1,PARCEL_014,2026-01-27,0.457,27.6,0.0,OK
2,PARCEL_006,2026-05-17,0.497,25.8,0.0,OK
3,PARCEL_004,10/02/2026,0.810,25.0,0.0,OK
4,PARCEL_002,2026-01-17,0.269,33.2,0.0,OK


In [156]:
# Duplicate checks
reading_duplicates = readings.duplicated(subset=["parcel_id", "date", "ndvi_value"]).sum()
metadata_duplicates = metadata.duplicated(subset=["parcel_id"]).sum()

print(f"Duplicate reading rows: {reading_duplicates}")
print(f"Duplicate metadata rows: {metadata_duplicates}")

Duplicate reading rows: 0
Duplicate metadata rows: 0


In [157]:
# ── Issue 1: Mixed date formats → parse to a single datetime ──────────
# Three formats found: YYYY-MM-DD, DD/MM/YYYY, DD-Mon-YYYY
readings["date"] = pd.to_datetime(readings["date"],format="mixed",dayfirst=True)
metadata["sowing_date"] = pd.to_datetime(metadata["sowing_date"], format="mixed",dayfirst=True)

In [158]:
# ── Issue 2: sensor_status has inconsistent case + whitespace ─────────
# Values seen: 'OK', 'ok', 'OK ', ' OK', 'Error', 'ERROR', 'error'
# Normalise to lowercase, strip whitespace → 'ok' or 'error'
readings["sensor_status"] = readings["sensor_status"].str.strip().str.lower()

In [159]:
# ── Issue 3: sensor_status NaN (137 rows) → treat as 'unknown' ────────
# We cannot call these 'ok'; flagging lets analysis exclude or include.
readings["sensor_status"] = readings["sensor_status"].fillna("unknown")

In [160]:
# ── Issue 4: NDVI values outside valid range [-1, 1] (104 rows) ───
# These are sensor/transmission errors — cannot impute meaningfully
# without neighbouring-day context (out of scope here). Flag and nullify.
invalid_ndvi_mask = (readings["ndvi_value"] < -1) | (readings["ndvi_value"] > 1)
print(f"NDVI out-of-range count: {invalid_ndvi_mask.sum()} rows")
readings.loc[invalid_ndvi_mask, "ndvi_value"] = np.nan
readings["ndvi_flag"] = invalid_ndvi_mask.map({True: "out_of_range", False: "ok"})
print(f"NDVI out-of-range nullified: {invalid_ndvi_mask.sum()} rows")

NDVI out-of-range count: 104 rows
NDVI out-of-range nullified: 104 rows


### Joining Reading and Metadata

In [161]:
joined = readings.merge(metadata, on="parcel_id", how="inner")

In [162]:
joined.head()

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,ndvi_flag,mill_id,crop_type,sowing_date,area_hectares
0,PARCEL_018,2026-05-16,0.595,15.4,0.0,error,ok,MILL_SOUTH,sugarcane,2026-11-01,5.82
1,PARCEL_014,2026-01-27,0.457,27.6,0.0,ok,ok,MILL_NORTH,sugarcane,2026-05-01,9.39
2,PARCEL_006,2026-05-17,0.497,25.8,0.0,ok,ok,MILL_WEST,sugarcane,2026-11-02,5.67
3,PARCEL_004,2026-02-10,0.810,25.0,0.0,ok,ok,MILL_NORTH,sugarcane,2026-02-01,3.18
4,PARCEL_002,2026-01-17,0.269,33.2,0.0,ok,ok,MILL_SOUTH,sugarcane,2026-01-16,8.97


### Analysis

In [163]:
"""
For each crop_type, compute mean NDVI in the 30 days before and after
sowing_date. Excludes rows where sensor_status is not 'ok'.
"""
# Use only clean sensor readings
clean = joined[joined["sensor_status"] == "ok"].copy()

# Days relative to sowing
clean["days_from_sowing"] = (clean["date"] - clean["sowing_date"]).dt.days

before = (
    clean[(clean["days_from_sowing"] >= -30) & (clean["days_from_sowing"] < 0)]
    .groupby("crop_type")
    .agg(
        mean_ndvi_before=("ndvi_value", "mean"),
        n_parcels_before=("parcel_id", "nunique"),
    )
)

after = (
    clean[(clean["days_from_sowing"] > 0) & (clean["days_from_sowing"] <= 30)]
    .groupby("crop_type")
    .agg(
        mean_ndvi_after=("ndvi_value", "mean"),
        n_parcels_after=("parcel_id", "nunique"),
    )
)

result = before.join(after, how="outer")

# n_parcels = parcels that appear in at least one of the two windows
result["n_parcels"] = result[["n_parcels_before", "n_parcels_after"]].max(axis=1).astype(int)
result = result[["mean_ndvi_before", "mean_ndvi_after", "n_parcels"]].round(4)
result = result.reset_index()

In [164]:
display(result)

,crop_type,mean_ndvi_before,mean_ndvi_after,n_parcels
0,soybean,0.2312,0.3286,4
1,sugarcane,0.3327,0.3936,19
2,wheat,0.2058,0.3263,2
